# Data Preparation

In this notebook, we focus on preparing the datasets provided for the subsequent clustering analysis.  
The goal is to obtain a clean and unified dataset that is suitable for unsupervised learning.

This data preparation phase involves an initial exploration of the dataset structure and the identification of features that are relevant for clustering, with particular attention to numerical variables. The process includes cleaning the data by handling missing values, removing values outside the expected feature domains, and detecting and treating outliers. Throughout this process, we aim to preserve as much informative content as possible, in order to retain the distinctive characteristics of the original data.

We begin by importing an already **enriched** version of the dataset, where by enriched, we refer to a dataset in which some missing or incomplete information has been previously reintegrated, as described in detail in the project documentation.

---

In [1]:
import pandas as pd

tracks = pd.read_csv("../../enriched_datasets/tracks_enriched.csv")
artists = pd.read_csv("../../enriched_datasets/artists.csv")

tracks = tracks.copy()
artists = artists.copy()

print(f"Initial dataset info:\n")
tracks.info()
artists.info()

# Print original number of tracks and artists
print(f"Tracks shape: {tracks.shape[0]} rows x {tracks.shape[1]} columns")
print(f"Artists shape: {artists.shape[0]} rows x {artists.shape[1]} columns")

Initial dataset info:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11166 entries, 0 to 11165
Data columns (total 45 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   id                    11166 non-null  object 
 1   id_artist             11166 non-null  object 
 2   name_artist           11166 non-null  object 
 3   full_title            11166 non-null  object 
 4   title                 11166 non-null  object 
 5   featured_artists      3517 non-null   object 
 6   primary_artist        11166 non-null  object 
 7   language              11061 non-null  object 
 8   album                 9652 non-null   object 
 9   stats_pageviews       4642 non-null   float64
 10  swear_IT              11166 non-null  int64  
 11  swear_EN              11166 non-null  int64  
 12  swear_IT_words        11166 non-null  object 
 13  swear_EN_words        11166 non-null  object 
 14  year                  10951 non-null  object 
 

## Casting Data into Correct Types

In this step, we convert track's and artist's related features to their appropriate data types to ensure consistency and correctness in subsequent analyses.

In [2]:
# Objects to strings
columns_to_string_tracks = ["id", "id_artist", "name_artist", "full_title", "title", "featured_artists", "primary_artist", "language", "album", "album_name", "album_type", "lyrics", "album_image", "id_album"]
for column in columns_to_string_tracks:
    tracks[column] = tracks[column].astype("string")

# Album release date: object -> datetime
tracks["album_release_date"] = pd.to_datetime(tracks["album_release_date"], errors="coerce")

# Floats/Objects to Ints
columns_to_ints_tracks = ["year", "month", "day", "popularity"]
for column in columns_to_ints_tracks:
    tracks[column] = pd.to_numeric(tracks[column], errors="coerce")
    tracks[column] = tracks[column].astype("Int64")

# Explicit: object -> boolean
print(tracks["explicit"].head())
tracks["explicit"] = tracks["explicit"].astype("bool")

print(f"Tracks after correct casting:\n")
tracks.info()

0    True
1    True
2    True
3    True
4    True
Name: explicit, dtype: object
Tracks after correct casting:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11166 entries, 0 to 11165
Data columns (total 45 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   id                    11166 non-null  string        
 1   id_artist             11166 non-null  string        
 2   name_artist           11166 non-null  string        
 3   full_title            11166 non-null  string        
 4   title                 11166 non-null  string        
 5   featured_artists      3517 non-null   string        
 6   primary_artist        11166 non-null  string        
 7   language              11061 non-null  string        
 8   album                 9652 non-null   string        
 9   stats_pageviews       4642 non-null   float64       
 10  swear_IT              11166 non-null  int64         
 11  swear_EN             

In [3]:
columns_to_string_artists = ["id_author", "name", "gender", "birth_place", "nationality", "description", "province", "region", "country", "source"]
for column in columns_to_string_artists:
    artists[column] = artists[column].astype("string")
    
columns_to_datetime_artists = ["birth_date", "active_start", "active_end"]
for column in columns_to_datetime_artists:
    artists[column] = pd.to_datetime(artists[column], errors='coerce')

print(f"Artists after correct casting:\n")
artists.info()

Artists after correct casting:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 104 entries, 0 to 103
Data columns (total 15 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   id_author     104 non-null    string        
 1   name          104 non-null    string        
 2   gender        97 non-null     string        
 3   birth_date    88 non-null     datetime64[ns]
 4   birth_place   98 non-null     string        
 5   nationality   103 non-null    string        
 6   description   104 non-null    string        
 7   active_start  65 non-null     datetime64[ns]
 8   active_end    0 non-null      datetime64[ns]
 9   province      93 non-null     string        
 10  region        93 non-null     string        
 11  country       102 non-null    string        
 12  latitude      100 non-null    float64       
 13  longitude     100 non-null    float64       
 14  source        31 non-null     string        
dtypes: datet

---

## Removing Duplicates

We observe that some track identifiers appear multiple times in the dataset while referring to different songs. For this reason, duplicates are identified using more informative criteria rather than relying solely on track IDs.

Specifically, we perform the following checks:
- we verify whether the _id_author_ field in the artists dataset contains duplicate entries;
- within the tracks dataset, we identify duplicates as tracks sharing both the same _id_artist_ and _full_title_.

After identifying duplicate records, we remove them by retaining only the first occurrence of each duplicated entry.


In [4]:
duplicate_tracks = tracks[tracks.duplicated(subset=["id"], keep=False)]
print(f"Duplicated track ids:\n", duplicate_tracks)
print("Same track ids but different song titles.")

print ("\n")

# IDs appearing more than once
duplicated_ids = artists["id_author"].value_counts()[artists["id_author"].value_counts() > 1]
print(f"Dupicated author ids:\n", duplicated_ids)

print("\n")

duplicate_pairs = tracks[tracks.duplicated(subset=["id_artist", "full_title"], keep=False)]
print(f"Duplicated tracks:\n", duplicate_pairs)

# Keeping only first ID and full_title of the duplicates
tracks = tracks.drop_duplicates(subset=["id_artist", "full_title"], keep="first")

print("\n")

print(f"Tracks shape after removing duplicates: {tracks.shape[0]} rows x {tracks.shape[1]} columns")
print(f"Artists shape after removing duplicates: {artists.shape[0]} rows x {artists.shape[1]} columns")

Duplicated track ids:
              id    id_artist    name_artist  \
43     TR715264  ART04205421  Rosa Chemical   
120    TR976686  ART19605256           Beba   
141    TR230274  ART18853907           Alfa   
159    TR531651  ART18853907           Alfa   
199    TR898853  ART88026810         thasup   
...         ...          ...            ...   
10879  TR292480  ART07024718          Fedez   
10915  TR978886  ART07024718          Fedez   
10962  TR925275  ART07024718          Fedez   
11007  TR747430  ART02733420      Marracash   
11160  TR458543  ART02733420      Marracash   

                                              full_title  \
43                       ​non è normale by Rosa Chemical   
120                                       Ibridi by Beba   
141                   SaN LoREnZo by Alfa (Ft. Annalisa)   
159    Serenata - From “Forever Out of My League” by ...   
199                         ​oh 9od by thasup (Ft. nayt)   
...                                                 

---

## Removing Non-Numeric Features

With the clustering goal in mind, we remove all non-numeric features, with the following exceptions:
- the _id_artist_ and _id_author_ columns are retained to enable the subsequent merging of the two datasets;
- the _title_ is retained for a subsequent study on the clustering;
- _datetime_ features are preserved in order to later extract temporal components such as day, month, or year.

This step is performed **prior to handling missing values**, so as to avoid removing observations due to non-informative or irrelevant features.


In [5]:
# TRACKS
numeric_cols_t = tracks.select_dtypes(include=["number"]).columns
for col in numeric_cols_t:
    tracks[col] = pd.to_numeric(tracks[col], errors='coerce')

reduced_tracks = tracks[numeric_cols_t]

# Inserting artist id (string)
if "id_artist" in tracks.columns:
    reduced_tracks.insert(0, "id_artist", tracks["id_artist"])

# Inserting title (string)
if "title" in tracks.columns:
    reduced_tracks.insert(0, "title", tracks["title"])

# Inserting album_release (datetime)
if "album_release_date" in tracks.columns:
    reduced_tracks.insert(1, "album_release_date", tracks["album_release_date"])

print(f"Tracks shape after removing non-numeric features: {reduced_tracks.shape[0]} rows x {reduced_tracks.shape[1]} columns")

Tracks shape after removing non-numeric features: 11164 rows x 29 columns


In [6]:
# ARTISTS
numeric_cols_a = artists.select_dtypes(include=["number"]).columns
for col in numeric_cols_a:
    artists[col] = pd.to_numeric(artists[col], errors='coerce')

reduced_artists = artists[numeric_cols_a]

# Inserting author id
if "id_author" in artists.columns:
    reduced_artists.insert(0, "id_author", artists["id_author"])

# Inserting datetimes
if "birth_date" in artists.columns:
    reduced_artists.insert(1, "birth_date", artists["birth_date"])

if "active_start" in artists.columns:
    reduced_artists.insert(2, "active_start", artists["active_start"])

if "active_end" in artists.columns:
    reduced_artists.insert(3, "active_end", artists["active_end"])

print(f"Artists shape after removing non-numeric features: {reduced_artists.shape[0]} rows x {reduced_artists.shape[1]} columns")

Artists shape after removing non-numeric features: 104 rows x 6 columns


---

## Missing Values Management

For both the tracks and artists datasets, we adopt strategies aimed at minimizing information loss, avoiding the removal of entire features or an excessive number of observations.

The adopted approaches are described separately for each dataset.

### Missing Values in the Tracks Dataset

To avoid either discarding the _album_release_date_ feature or losing more than 300 observations, we extract the year component from the full release date and use the _year_ feature to replace its missing values in the corresponding entries. This choice is justified by the fact that some tracks may have been released as singles, for which an album release date may not be available.

After this transformation, we drop the _album_release_date_ feature and retain only the derived _album_release_year_ variable.

We then arbitrarily set a threshold of 300 missing values to determine which features to retain; features exceeding this threshold are removed. Finally, all remaining observations containing at least one missing value are discarded.

In [7]:
reduced_tracks["album_release_year"] = (
    reduced_tracks["album_release_date"].dt.year
    .fillna(reduced_tracks["year"])
)

reduced_tracks = reduced_tracks.drop(columns=["album_release_date"])

tracks_nan_per_feature = reduced_tracks.isna().sum()
print(f"Nan per feature (tracks):", tracks_nan_per_feature)

print("\n")

tracks_nan_per_row = reduced_tracks.isna().sum(axis=1)
print(f"Nan per row:", tracks_nan_per_row)

print("\n")

# Removing columns with more than 300 nans
cols_to_drop_tracks = tracks_nan_per_feature[tracks_nan_per_feature > 300].index
reduced_tracks = reduced_tracks.drop(columns=cols_to_drop_tracks)

print("Dropped columns for tracks:", list(cols_to_drop_tracks))

# Dropping rows with at least one nan
reduced_tracks = reduced_tracks.dropna()

print("n")

# Checking remaining tracks
print(f"Tracks shape after removing nans: {reduced_tracks.shape[0]} rows x {reduced_tracks.shape[1]} columns")

Nan per feature (tracks): title                      0
id_artist                  0
stats_pageviews         6524
swear_IT                   0
swear_EN                   0
year                     237
month                    800
day                      834
n_sentences               76
n_tokens                  76
tokens_per_sent           76
char_per_tok              76
lexical_density           76
avg_token_per_clause      76
bpm                       64
centroid                  64
rolloff                   64
flux                      64
rms                       64
zcr                       64
flatness                  64
spectral_complexity       64
pitch                     64
loudness                  64
disc_number               78
track_number              78
duration_ms               78
popularity                29
album_release_year         5
dtype: int64


Nan per row: 0        0
1        0
2        0
3        0
4        0
        ..
11161    4
11162    1
11163    4
11164 

/tmp/ipykernel_236767/3014946735.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  reduced_tracks["album_release_year"] = (


### Missing Values in the Artists Dataset

Similarly to the previous case, the _active_start_ feature contains a large number of missing values. To address this issue, we replace missing entries with the year of publication of the oldest track associated with the same artist. As before, we retain only the year component, stored as the _active_start_year_ feature.

At this stage, we also remove the _year_ feature from the tracks dataset, as the information it provides would be redundant with respect to the temporal features previously extracted.

For the artists dataset, we remove only features containing more than 20 missing values. Finally, all remaining observations with at least one missing value are discarded.

In [8]:
# Active start year
first_year_by_artist = reduced_tracks.groupby("id_artist")["year"].min()
first_year_by_artist.name = "first_track_year"

print(first_year_by_artist.head())

reduced_artists = reduced_artists.merge(
    first_year_by_artist,
    left_on="id_author",
    right_on="id_artist",
    how="left"
)

print(reduced_artists.columns)

reduced_artists["active_start_year"] = (
    reduced_artists["active_start"].dt.year
    .fillna(reduced_artists["first_track_year"])
)

reduced_artists = reduced_artists.drop(columns=["active_start"])
reduced_tracks = reduced_tracks.drop(columns=["year"])
reduced_artists = reduced_artists.drop(columns=["first_track_year"])

print("\n")

artists_nan_per_feature = reduced_artists.isna().sum()
print(f"Nan per feature (artists):\n", artists_nan_per_feature)

print("\n")

artists_nan_per_row = reduced_artists.isna().sum(axis=1)
print(f"Nan per row (artists):\n", artists_nan_per_row)

print("\n")

# Removing columns with more than 20 nans
cols_to_drop_artists = artists_nan_per_feature[artists_nan_per_feature > 20].index
reduced_artists = reduced_artists.drop(columns=cols_to_drop_artists)

print("Dropped columns for artists:", list(cols_to_drop_artists))

# Dropping rows with at least one nan
reduced_artists = reduced_artists.dropna()

print("\n")

#Checking remaining artists
print(f"Artists shape after removing nans: {reduced_artists.shape[0]} rows x {reduced_artists.shape[1]} columns")

id_artist
ART02449272    2016
ART02666525    2019
ART02733420    1947
ART03111237    2001
ART04141409    1900
Name: first_track_year, dtype: Int64
Index(['id_author', 'birth_date', 'active_start', 'active_end', 'latitude',
       'longitude', 'first_track_year'],
      dtype='object')


Nan per feature (artists):
 id_author              0
birth_date            16
active_end           104
latitude               4
longitude              4
active_start_year      0
dtype: int64


Nan per row (artists):
 0      2
1      1
2      1
3      2
4      1
      ..
99     1
100    1
101    1
102    1
103    1
Length: 104, dtype: int64


Dropped columns for artists: ['active_end']


Artists shape after removing nans: 88 rows x 5 columns


In [9]:
# Final check on all nans (should both be zero)
print(f"Making sure we dropped nans.")

tracks_nan_per_feature = reduced_tracks.isna().sum()
print(f"Nan per feature after nan removal(tracks):", tracks_nan_per_feature)

print("\n")

artists_nan_per_feature = reduced_artists.isna().sum()
print(f"Nan per feature after nan removal(artists):", artists_nan_per_feature)


Making sure we dropped nans.
Nan per feature after nan removal(tracks): title                   0
id_artist               0
swear_IT                0
swear_EN                0
n_sentences             0
n_tokens                0
tokens_per_sent         0
char_per_tok            0
lexical_density         0
avg_token_per_clause    0
bpm                     0
centroid                0
rolloff                 0
flux                    0
rms                     0
zcr                     0
flatness                0
spectral_complexity     0
pitch                   0
loudness                0
disc_number             0
track_number            0
duration_ms             0
popularity              0
album_release_year      0
dtype: int64


Nan per feature after nan removal(artists): id_author            0
birth_date           0
latitude             0
longitude            0
active_start_year    0
dtype: int64


---

## Invalid Values Removal

For all features with a well-defined domain, we remove values that fall outside the expected range.  
This procedure is applied to both the tracks and artists datasets.

The domain choices adopted in this step are documented in detail in the project report and are summarized below:
- **popularity** → [0, 100];
- **album_release_year** → ≤ 2025 (values in the future are not admissible);
- **zcr**, **flatness** → [0, 1];
- **active_start_year** → greater than **birth_date_year**;
- **latitude** → [35, 47];
- **longitude** → [4, 19].

It is worth noting that not all potentially incorrect values are automatically removed. For instance, artists with latitude or longitude values outside the specified ranges may have been born outside Italy while still holding Italian citizenship. For this reason, such checks are used primarily to identify anomalous cases rather than to enforce strict filtering.

Some validations are therefore performed only for exploratory purposes, in order to highlight unexpected or noteworthy values. In particular, in the final outputs of the following cell we inspect the minimum and maximum values of selected features to assess their plausibility and potential relevance.

In [10]:
# popularity
reduced_tracks = reduced_tracks[
    (reduced_tracks["popularity"] >= 0) &
    (reduced_tracks["popularity"] <= 100)
]

print(f"Checking min popularity:", reduced_tracks["popularity"].min())
print(f"Checking max popularity:", reduced_tracks["popularity"].max())

print(f"Tracks shape after removing invalid popularity: {reduced_tracks.shape[0]} rows x {reduced_tracks.shape[1]} columns")

Checking min popularity: 0
Checking max popularity: 100
Tracks shape after removing invalid popularity: 10790 rows x 25 columns


In [11]:
# swear_IT
count_invalid_swear_IT = (reduced_tracks["swear_IT"] < 0).sum()
print(f"Invalid swear_IT counter:", count_invalid_swear_IT)

print("\n")

# swear_EN
count_invalid_swear_EN = (reduced_tracks["swear_EN"] < 0).sum()
print(f"Invalid swear_EN counter:", count_invalid_swear_EN)

print("\n")

Invalid swear_IT counter: 0


Invalid swear_EN counter: 0




In [12]:
# album_release_year
reduced_tracks = reduced_tracks[reduced_tracks["album_release_year"] <= 2025]

print(f"Tracks shape after removing invalid album release year: {reduced_tracks.shape[0]} rows x {reduced_tracks.shape[1]} columns")

Tracks shape after removing invalid album release year: 10788 rows x 25 columns


In [13]:
# zcr
count_invalid_zcr = ((reduced_tracks["zcr"] < 0) | (reduced_tracks["zcr"] > 1)).sum()
print(f"Invalid number of zcr:", count_invalid_zcr)

print("\n")

# flatness
count_invalid_flatness = ((reduced_tracks["flatness"] < 0) | (reduced_tracks["flatness"] > 1)).sum()
print(f"Invalid number of flatness:", count_invalid_flatness)

Invalid number of zcr: 0


Invalid number of flatness: 0


In [14]:
# active_start_year
reduced_artists["birth_date_year"] = (
    reduced_artists["birth_date"].dt.year
)

reduced_artists = reduced_artists.drop(columns=["birth_date"])

count_invalid_active_start = (reduced_artists["active_start_year"] <  reduced_artists["birth_date_year"]).sum()
print(f"Invalid number of active start year:", count_invalid_active_start)

mask_invalid_active_start = reduced_artists["active_start_year"] < reduced_artists["birth_date_year"]

reduced_artists.loc[mask_invalid_active_start, "active_start_year"] = (
    reduced_artists["birth_date_year"] + 18
)
print("Also dropping brith date year because of high correlation:")
reduced_artists = reduced_artists.drop(columns=["birth_date_year"])

print("\n")

# latitude
count_invalid_latitude = ((reduced_artists["latitude"] < 35) | (reduced_artists["latitude"] > 47)).sum()
print(f"Invalid number of latitude:", count_invalid_latitude)

invalid_latitude = reduced_artists[
    (reduced_artists["latitude"] <= 35) |
    (reduced_artists["latitude"] >= 47)
]

print(f"Artists with invalid latitude:", invalid_latitude[["id_author", "latitude", "longitude"]])
print("Checked in dataset: born in Santo Domingo, italian.")

print("\n")

# longitude
count_invalid_longitude = ((reduced_artists["longitude"] < 4) | (reduced_artists["longitude"] > 19)).sum()
print(f"Invalid number of longitude:", count_invalid_longitude)

invalid_longitude = reduced_artists[
    (reduced_artists["longitude"] <= 4) |
    (reduced_artists["longitude"] >= 19)
]

print(f"Artists with invalid longitude:", invalid_longitude[["id_author", "latitude", "longitude"]])
print("Checked in dataset: born in Santo Domingo, italian.")

Invalid number of active start year: 20
Also dropping brith date year because of high correlation:


Invalid number of latitude: 1
Artists with invalid latitude:       id_author  latitude  longitude
31  ART71515715   18.4861   -69.9312
Checked in dataset: born in Santo Domingo, italian.


Invalid number of longitude: 1
Artists with invalid longitude:       id_author  latitude  longitude
31  ART71515715   18.4861   -69.9312
Checked in dataset: born in Santo Domingo, italian.


In [15]:
# disc_number
print(f"Disc number min value:", reduced_tracks["disc_number"].min())
print(f"Disc number max value:", reduced_tracks["disc_number"].max())

print("\n")

#track_number
print(f"Track number min value:", reduced_tracks["track_number"].min())
print(f"Track number max value:", reduced_tracks["track_number"].max())

print("\n")

#duration_ms
print(f"Duration min value:", reduced_tracks["duration_ms"].min())
print(f"Duration max value:", reduced_tracks["duration_ms"].max())

Disc number min value: 1.0
Disc number max value: 5.0


Track number min value: 1.0
Track number max value: 54.0


Duration min value: 11426.0
Duration max value: 3753057.0


---

## Outlier Removal

Outlier handling is divided into four main categories, each requiring different assumptions and treatment strategies.

### Sound Features

Since all the considered **sound features** have a theoretical domain of $[0, +\infty)$, we first ensure that no values violate the non-negativity constraint.

Outliers are then handled using the *sigma-based* (standard deviation) method, which we consider more flexible than the IQR approach for this specific set of features.

### Year Features for Tracks

For the _album_release_year_ feature, an upper bound has already been defined during the invalid values removal phase. Consequently, outlier detection is applied only to the lower bound, in order to identify implausibly early release years.

### Lexical Features

The same general methodology is applied to lexical features: we first ensure that all values are non-negative, and then explore the acceptable range for each feature.

Given the possibility of longer tracks (e.g. rap ballads) or cases in which a single character may be considered a valid word, we deliberately avoid removing certain extreme values, as they may carry meaningful information for clustering.

To guide this decision, we inspect the minimum and maximum values of each feature, identify those affected by particularly strong outliers, and selectively apply cuts only where deemed appropriate.

### Year Features for Artists

Finally, we evaluate outliers for artist related year features, including _active_start_year_. Since all observed values appear reasonable, no outlier removal is performed for these features.


In [16]:
# === 1. ===
sound_outlier_mask_total = pd.Series(False, index=reduced_tracks.index)

sound_cols = ["bpm", "centroid", "rolloff", "flux", "rms", "spectral_complexity", "pitch", "loudness"]
for feature in sound_cols:
    col = reduced_tracks[feature]

    min = col.min()
    max = col.max()
    median = col.median()
    mu = col.mean()
    sigma = col.std()

    lower = 0
    upper = mu + 4 * sigma

    outlier_mask = (col < lower) | (col > upper)
    n_outliers = outlier_mask.sum()

    print("="*40)
    print(f"Feature: {feature}")
    print(f"Min value: {min}")
    print(f"Max value: {max}")
    print(f"Mean value: {mu}")
    print(f"Median value:{median}")
    print(f"Acceptable domain (4*sigma): [{lower:.3f}, {upper:.3f}]")
    print(f"Outliers found: {n_outliers}")

    sound_outlier_mask_total |= outlier_mask

# Counts total of outliers
print("Sound outliers rows:", sound_outlier_mask_total.sum())

# Removing outliers rows
reduced_tracks = reduced_tracks[~sound_outlier_mask_total]

print("\n")
print(f"Tracks shape after sound outliers removal: {reduced_tracks.shape[0]} rows x {reduced_tracks.shape[1]} columns")
print("\n")

Feature: bpm
Min value: 59.97
Max value: 738.27
Mean value: 114.11265850945493
Median value:106.95
Acceptable domain (4*sigma): [0.000, 221.575]
Outliers found: 1
Feature: centroid
Min value: 0.0
Max value: 0.2982
Mean value: 0.13790420837968115
Median value:0.1374
Acceptable domain (4*sigma): [0.000, 0.249]
Outliers found: 4
Feature: rolloff
Min value: 0.0
Max value: 8635.9542
Mean value: 1620.040781989247
Median value:1553.15255
Acceptable domain (4*sigma): [0.000, 3882.805]
Outliers found: 20
Feature: flux
Min value: 0.0
Max value: 1.9285
Mean value: 1.2594609658880236
Median value:1.2573
Acceptable domain (4*sigma): [0.000, 1.805]
Outliers found: 4
Feature: rms
Min value: 0.0
Max value: 0.6219
Mean value: 0.224961957730812
Median value:0.2303
Acceptable domain (4*sigma): [0.000, 0.481]
Outliers found: 3
Feature: spectral_complexity
Min value: 0.0
Max value: 61.2225
Mean value: 27.54860963107156
Median value:27.44175
Acceptable domain (4*sigma): [0.000, 61.085]
Outliers found: 1
Fea

In [17]:
# === 2. ===
tracks_year_outlier_mask_total = pd.Series(False, index=reduced_tracks.index)

tracks_datetime_cols = ["album_release_year"]
for feature in tracks_datetime_cols:
    col = reduced_tracks[feature]

    min = col.min()
    max = col.max()
    median = col.median()
    mu = col.mean()
    sigma = col.std()

    lower = mu - 5 * sigma
    upper = 2025

    outlier_mask = (col < lower) | (col > upper)
    n_outliers = outlier_mask.sum()

    print("="*40)
    print(f"Feature: {feature}")
    print(f"Min value: {min}")
    print(f"Max value: {max}")
    print(f"Median value:{median}")
    print(f"Acceptable domain (5*sigma): [{lower:.3f}, {upper:.3f}]")
    print(f"Outliers found: {n_outliers}")

    tracks_year_outlier_mask_total |= outlier_mask

# Counts total of outliers
print("Tracks year outliers rows:", tracks_year_outlier_mask_total.sum())

# Removing outliers rows
reduced_tracks = reduced_tracks[~tracks_year_outlier_mask_total]

print("\n")
print(f"Tracks shape after year outliers removal: {reduced_tracks.shape[0]} rows x {reduced_tracks.shape[1]} columns")
print("\n")

Feature: album_release_year
Min value: 1905.0
Max value: 2025.0
Median value:2018.0
Acceptable domain (5*sigma): [1978.870, 2025.000]
Outliers found: 16
Tracks year outliers rows: 16


Tracks shape after year outliers removal: 10737 rows x 25 columns




In [18]:
# === 3. ===
lexical_cols = ["n_sentences", "n_tokens", "tokens_per_sent", "char_per_tok", "lexical_density", "avg_token_per_clause"]

# Choosing only some columns to clean
cols_to_clean = ["tokens_per_sent", "avg_token_per_clause"]

for feature in lexical_cols:
    col = reduced_tracks[feature]

    mu = col.mean()
    sigma = col.std()
    lower = mu - 5 * sigma
    upper = mu + 5 * sigma

    outlier_mask = (col < lower) | (col > upper)
    n_outliers = outlier_mask.sum()

    print("="*40)
    print(f"Feature: {feature}")
    print(f"Min value: {col.min()}")
    print(f"Max value: {col.max()}")
    print(f"Median value: {col.median()}")
    print(f"Acceptable domain (5*sigma): [{lower:.3f}, {upper:.3f}]")
    print(f"Outliers found: {n_outliers}")

    # Outlier removal only on chosen columns
    if feature in cols_to_clean:
        reduced_tracks = reduced_tracks[~outlier_mask]
        print(f"Removed {n_outliers} outliers from {feature}")
    else:
        print("Skipped outlier removal for this feature.")

print("\n")
print(f"Tracks shape after lexical outliers removal: {reduced_tracks.shape[0]} rows x {reduced_tracks.shape[1]} columns")
print("\n")

Feature: n_sentences
Min value: 1.0
Max value: 437.0
Median value: 59.0
Acceptable domain (5*sigma): [-62.663, 182.392]
Outliers found: 18
Skipped outlier removal for this feature.
Feature: n_tokens
Min value: 3.0
Max value: 3089.0
Median value: 495.0
Acceptable domain (5*sigma): [-534.235, 1536.460]
Outliers found: 15
Skipped outlier removal for this feature.
Feature: tokens_per_sent
Min value: 1.5
Max value: 283.0
Median value: 8.41304347826087
Acceptable domain (5*sigma): [-12.928, 30.217]
Outliers found: 36
Removed 36 outliers from tokens_per_sent


Feature: char_per_tok
Min value: 2.0
Max value: 12.0
Median value: 4.013377926421405
Acceptable domain (5*sigma): [1.829, 6.280]
Outliers found: 77
Skipped outlier removal for this feature.
Feature: lexical_density
Min value: 0.0
Max value: 1.0
Median value: 0.51171875
Acceptable domain (5*sigma): [0.210, 0.819]
Outliers found: 52
Skipped outlier removal for this feature.
Feature: avg_token_per_clause
Min value: 0.0
Max value: 660.0
Median value: 6.764705882352941
Acceptable domain (5*sigma): [-64.286, 80.154]
Outliers found: 24
Removed 24 outliers from avg_token_per_clause


Tracks shape after lexical outliers removal: 10677 rows x 25 columns




In [19]:
# === 4. ===
artists_datetime_cols = ["active_start_year"]
for feature in artists_datetime_cols:
    col = reduced_artists[feature]

    min = col.min()
    max = col.max()
    median = col.median()
    mu = col.mean()
    sigma = col.std()

    lower = mu - 5 * sigma
    upper = 2025

    outlier_mask = (col < lower) | (col > upper)
    n_outliers = outlier_mask.sum()

    print("="*40)
    print(f"Feature: {feature}")
    print(f"Min value: {min}")
    print(f"Max value: {max}")
    print(f"Median value:{median}")
    print(f"Acceptable domain (5*sigma): [{lower:.3f}, {upper:.3f}]")
    print(f"Outliers found: {n_outliers}")

print("\n")
print("No need to remove anything.")

Feature: active_start_year
Min value: 1988.0
Max value: 2021.0
Median value:2007.0
Acceptable domain (5*sigma): [1960.762, 2025.000]
Outliers found: 0


No need to remove anything.


---

## Assessment of the Final Datasets

Before saving the cleaned datasets, we perform a quick assessment to verify their structure and content. Specifically, we examine:

- The shape of both the tracks and artists datasets;
- The first few rows of each dataset using `.head()`;
- Summary information of each dataset using `.info()`.

In [20]:
print(f"Tracks shape after data preparation: {reduced_tracks.shape[0]} rows x {reduced_tracks.shape[1]} columns")
print(f"Artists shape after after data preparation: {reduced_artists.shape[0]} rows x {reduced_artists.shape[1]} columns")

print("\n")

print(f"Tracks sample:\n", reduced_tracks.head())
print(f"Artists sample:\n", reduced_artists.head())

print("\n")

print(f"Tracks types:", reduced_tracks.info())
print(f"Artists types:", reduced_artists.info())

Tracks shape after data preparation: 10677 rows x 25 columns
Artists shape after after data preparation: 88 rows x 4 columns


Tracks sample:
           title    id_artist  swear_IT  swear_EN  n_sentences  n_tokens  \
0  ​polka 2 :-/  ART04205421        13         6        102.0     911.0   
1         POLKA  ART04205421         9        12         56.0     675.0   
2  ​britney ;-)  ART04205421        16        12         88.0     758.0   
3           CEO  ART04205421         8         3         37.0     382.0   
4        LONDRA  ART04205421         1         0         48.0     429.0   

   tokens_per_sent  char_per_tok  lexical_density  avg_token_per_clause  ...  \
0         8.931373      4.170455         0.575284              8.133929  ...   
1        12.053571      4.280851         0.648936             12.500000  ...   
2         8.613636      4.075251         0.556856              8.422222  ...   
3        10.324324      4.023881         0.534328              6.701754  ...   
4     

In [21]:
reduced_artists.to_csv("../../prepared_datasets/artists.csv", index=False)
reduced_tracks.to_csv("../../prepared_datasets/tracks.csv", index=False)